# 10 — Todos os benchmarks

Generaliza o `00_prototype` para um registry de benchmarks e escreve os `.tex` do paper.
Cada benchmark declara: como carregar, o prompt do LLM 1, o espaço de rótulos, como verbalizar
o rótulo para o assessor, e o texto do construto que abre todo prompt do LLM 2.

**Invariantes do desenho, checados por asserção na seção 9:**

- O GEPA maximiza `ρ(hedge, g_m)`. `g_m` é a variância das `R` extrações do LLM 1. Nenhum
  rótulo humano entra no objetivo, no feedback reflexivo ou no prompt do mestre.
- `u_m` (desacordo humano) e `consensus` só aparecem na avaliação, depois de tudo otimizado.
- A última linha do prompt do assessor é fixa e igual em todos os benchmarks e condições.

**Saídas em `paper/`:** `tab_data.tex`, `tab_main.tex`, `tab_ablation.tex`, `tab_stability.tex`,
`codebooks.tex`, `macros.tex` e `results_skeleton.tex`.

## 1. Setup

In [ ]:
!pip install -q gepa datasets

import json, re, math, os, time, itertools, collections, hashlib, urllib.request
import requests, numpy as np, pandas as pd
from concurrent.futures import ThreadPoolExecutor
from scipy.stats import spearmanr
import gepa
from gepa.core.adapter import EvaluationBatch

try:
    from google.colab import userdata
    def secret(k): return userdata.get(k)
except Exception:
    def secret(k): return os.environ[k]

In [ ]:
CF_ACCOUNT = secret('CF_Palver_1')
CF_AUTH    = secret('CF_Palver_2')
GW = f"https://gateway.ai.cloudflare.com/v1/{CF_ACCOUNT}/default/workers-ai"
HEADERS = {"Content-Type": "application/json", "Authorization": CF_AUTH}
# sem cf-aig-skip-cache: prompt idêntico devolve resposta cacheada, o que torna
# reexecuções reprodutíveis apesar do não-determinismo de serving em temperatura 0.

ANNOTATOR_URL = f"{GW}/@cf/google/gemma-4-26b-a4b-it"
ASSESSOR_URL  = f"{GW}/@cf/zai-org/glm-4.7-flash"
MASTER_URL    = f"{GW}/@cf/zai-org/glm-5.2"
MASTER_EFFORT = "high"

HEDGES = ["not uncertain", "somewhat uncertain", "highly uncertain"]
H = len(HEDGES)
HEDGE_TO_ORD = {h: i for i, h in enumerate(HEDGES)}
FIXED_OUTPUT_LINE = ("On the LAST line, write EXACTLY one of the following and nothing else: "
                     + " | ".join(HEDGES) + ".")

WORKDIR = "./work"; PAPERDIR = "./paper"; RESDIR = "./results"
for d in (WORKDIR, PAPERDIR, RESDIR): os.makedirs(d, exist_ok=True)

SEED             = 42
R                = 20
TEMP_RESAMPLE    = 1.0
WORKERS          = 12
N_TRAIN_FOLDS    = 6
N_VAL_FOLDS      = 2
N_BOOT           = 2000
MAX_METRIC_CALLS = 60
COVERAGE_FLOOR   = 0.90
CONDITIONS       = ["labelled", "blind"]

## 2. Registry de benchmarks

Cada entrada devolve um DataFrame com `item_id, text, u_m, consensus, split_orig`. `text` já vem
formatado como o LLM 1 vai ler (o ConvAbuse traz o contexto do diálogo embutido, o AmbiStory traz
a narrativa e o sentido candidato).

In [ ]:
LEWIDI = ("https://raw.githubusercontent.com/Le-Wi-Di/le-wi-di.github.io/main/"
          "LeWiDi_2-2023/{d}_dataset/{d}_{split}.json")

def _lewidi(name, task_key=None):
    """task_key=None usa 'annotations'; senão puxa de other_info['other annotations']."""
    rows = []
    for split in ("train", "dev", "test"):
        with urllib.request.urlopen(LEWIDI.format(d=name, split=split)) as r:
            j = json.load(r)
        for k, v in j.items():
            raw = (v["annotations"] if task_key is None
                   else v["other_info"]["other annotations"][task_key])
            xs = raw.split(",")
            if any(x.strip() not in ("0", "1") for x in xs):
                continue                      # 3 itens do offensive trazem "No"
            a = [int(x) for x in xs]
            p = sum(a) / len(a)
            rows.append({"item_id": f"{name}-{split}-{k}", "text": v["text"], "n_ann": len(a),
                         "u_m": p * (1 - p), "consensus": int(p > 0.5), "split_orig": split})
    return pd.DataFrame(rows)

def load_convabuse():
    rows = []
    for split in ("train", "dev", "test"):
        with urllib.request.urlopen(LEWIDI.format(d="ConvAbuse", split=split)) as r:
            j = json.load(r)
        for k, v in j.items():
            a = [int(x) for x in v["annotations"].split(",")]
            p = sum(x == 1 for x in a) / len(a)          # 1 = não abusivo na escala original
            oi = v.get("other_info", {}) or {}
            ctx = "\n".join(x for x in [
                f"agent: {oi.get('previous_agent_turn','')}".strip(),
                f"user: {oi.get('previous_user_turn','')}".strip(),
                f"agent: {oi.get('agent_turn','')}".strip()] if x.split(": ", 1)[-1])
            rows.append({"item_id": f"ConvAbuse-{split}-{k}",
                         "text": (ctx + "\n" if ctx else "") + f"user: {v['text']}",
                         "n_ann": len(a), "u_m": p * (1 - p),
                         "consensus": int(p <= 0.5),      # abusivo = maioria não deu 1
                         "split_orig": split})
    return pd.DataFrame(rows)

AMBI = "https://raw.githubusercontent.com/Janosch-Gehring/ambistory/main/{}.json"

def load_ambistory():
    rows = []
    for split in ("train", "dev", "test"):
        with urllib.request.urlopen(AMBI.format(split)) as r:
            j = json.load(r)
        for k, v in j.items():
            ch = v["choices"]
            rows.append({
                "item_id": f"AmbiStory-{split}-{k}",
                "text": (f"{v['precontext']}\n{v['sentence']}\n{v['ending']}\n\n"
                         f"AMBIGUOUS WORD: {v['homonym']}\nCANDIDATE MEANING: {v['judged_meaning']}"),
                "n_ann": len(ch), "u_m": float(np.var(ch)),
                "consensus": int(round(np.mean(ch))), "split_orig": split})
    return pd.DataFrame(rows)

def load_mhs(item, n_max=4000):
    from datasets import load_dataset
    ds = load_dataset("ucberkeley-dlab/measuring-hate-speech", "default")["train"].to_pandas()
    need = {"comment_id", "text", item}
    missing = need - set(ds.columns)
    if missing:
        raise KeyError(f"colunas ausentes no MHS: {missing}. Disponíveis: {sorted(ds.columns)[:40]}")
    g = ds.groupby("comment_id").agg(text=("text", "first"), n_ann=(item, "size"),
                                     u_m=(item, "var"), mean=(item, "mean")).reset_index()
    g = g[g.n_ann >= 2].dropna(subset=["u_m"])
    g = g.sample(min(n_max, len(g)), random_state=SEED)
    rng = np.random.default_rng(SEED)
    r = rng.random(len(g))
    g["split_orig"] = np.where(r < .7, "train", np.where(r < .85, "dev", "test"))
    g["item_id"] = "MHS" + item + "-" + g.comment_id.astype(str)
    g["consensus"] = g["mean"].round().astype(int)
    return g[["item_id", "text", "n_ann", "u_m", "consensus", "split_orig"]]

In [ ]:
def _ann_prompt(unit, construct, labels_desc):
    return (f"You are coding {unit} for a research project.\n\n{construct}\n\n"
            f"Answer with exactly one character: {labels_desc}. Nothing else.")

BENCHMARKS = {
 "HSBrexit": dict(
   loader=lambda: _lewidi("HS-Brexit"), levels=[0,1],
   construct=("An English tweet about Brexit and immigration was labelled for HATE SPEECH: "
              "language that attacks, demeans or excludes people because of their religion, "
              "national origin or migration status."),
   ann=_ann_prompt("English tweets about Brexit and immigration. Placeholders <user> and <url> "
                   "stand for removed mentions and links",
                   "Label the tweet for HATE SPEECH: language that attacks, demeans or excludes "
                   "people because of their religion, national origin or migration status.",
                   "1 (hate speech) or 0 (not hate speech)"),
   verb={0:"not hate speech", 1:"hate speech"}),

 "HSBrexitOff": dict(
   loader=lambda: _lewidi("HS-Brexit", "offensive language detection"), levels=[0,1],
   construct=("An English tweet about Brexit and immigration was labelled for OFFENSIVE LANGUAGE: "
              "insults, slurs, profanity or contempt directed at a person or group."),
   ann=_ann_prompt("English tweets about Brexit and immigration",
                   "Label the tweet for OFFENSIVE LANGUAGE: insults, slurs, profanity or contempt "
                   "directed at a person or group.",
                   "1 (offensive) or 0 (not offensive)"),
   verb={0:"not offensive", 1:"offensive"}),

 "ArMIS": dict(
   loader=lambda: _lewidi("ArMIS"), levels=[0,1],
   construct=("An Arabic tweet was labelled for MISOGYNY OR SEXISM: content that demeans, "
              "stereotypes, polices or dismisses women, or treats them as inferior."),
   ann=_ann_prompt("Arabic tweets (Gulf and Levantine dialects)",
                   "Label the tweet for MISOGYNY OR SEXISM: content that demeans, stereotypes, "
                   "polices or dismisses women, or treats them as inferior.",
                   "1 (misogynistic or sexist) or 0 (not)"),
   verb={0:"not misogynistic", 1:"misogynistic or sexist"}),

 "ConvAbuse": dict(
   loader=load_convabuse, levels=[0,1],
   construct=("The last user turn of a conversation with a chatbot was labelled for ABUSE: "
              "sexist, racist, homophobic, transphobic or ableist language, sexual harassment, "
              "intellectual denigration, or hostility aimed at the system or at a group."),
   ann=_ann_prompt("turns from conversations between a human user and a chatbot",
                   "Label the LAST USER TURN for ABUSE: sexist, racist, homophobic, transphobic "
                   "or ableist language, sexual harassment, intellectual denigration, or hostility "
                   "aimed at the system, at a group, or at an individual.",
                   "1 (abusive) or 0 (not abusive)"),
   verb={0:"not abusive", 1:"abusive"}),

 "MDAgreement": dict(
   loader=lambda: _lewidi("MD-Agreement"), levels=[0,1],
   construct=("An English tweet from one of three domains (Black Lives Matter, the 2020 US "
              "election, Covid-19) was labelled for OFFENSIVENESS."),
   ann=_ann_prompt("English tweets from three domains: Black Lives Matter, the 2020 US election, "
                   "and Covid-19. Placeholders <user> and <url> stand for removed mentions and links",
                   "Label the tweet for OFFENSIVENESS: insults, slurs or profanity aimed at a "
                   "person or group, or language a reader could reasonably take as offensive.",
                   "1 (offensive) or 0 (not offensive)"),
   verb={0:"not offensive", 1:"offensive"}),

 "AmbiStory": dict(
   loader=load_ambistory, levels=[1,2,3,4,5],
   construct=("A short narrative contains an ambiguous word, and a candidate meaning for it was "
              "rated for PLAUSIBILITY on a 1-5 scale, from completely implausible to clearly "
              "intended."),
   ann=_ann_prompt("short narratives containing an ambiguous word",
                   "Rate how plausible it is that the ambiguous word carries the candidate meaning "
                   "in this story, from 1 (completely implausible) to 5 (clearly intended).",
                   "a single digit 1, 2, 3, 4 or 5"),
   verb={i: f"plausibility {i} of 5" for i in range(1, 6)}),

 "MHShatespeech": dict(
   loader=lambda: load_mhs("hatespeech"), levels=[0,1,2],
   construct=("A social media comment was judged for HATE SPEECH: bias-motivated, hostile and "
              "malicious language targeted at a person or group because of their actual or "
              "perceived innate characteristics."),
   ann=_ann_prompt("social media comments from YouTube, Reddit and Twitter",
                   "Does this comment contain hate speech, defined as bias-motivated, hostile and "
                   "malicious language targeted at a person or group because of their actual or "
                   "perceived innate characteristics?",
                   "0 (no), 1 (unclear) or 2 (yes)"),
   verb={0:"not hate speech", 1:"unclear", 2:"hate speech"}),

 "MHSdehumanize": dict(
   loader=lambda: load_mhs("dehumanize"), levels=[0,1,2,3,4],
   construct=("A social media comment was rated on whether it DEHUMANISES the group it refers to, "
              "for example by comparing them to an animal, on a 0-4 agreement scale."),
   ann=_ann_prompt("social media comments from YouTube, Reddit and Twitter",
                   "Rate agreement with: this comment dehumanises the group(s) it refers to (e.g. "
                   "by comparing them to an animal). 0 = strongly disagree, 4 = strongly agree.",
                   "a single digit 0, 1, 2, 3 or 4"),
   verb={i: f"dehumanisation rating {i} of 4" for i in range(5)}),

 "MHSattackdefend": dict(
   loader=lambda: load_mhs("attack_defend"), levels=[0,1,2,3,4],
   construct=("A social media comment was rated on whether it ATTACKS or DEFENDS the group it "
              "refers to, from 0 (strongly defending) to 4 (strongly attacking)."),
   ann=_ann_prompt("social media comments from YouTube, Reddit and Twitter",
                   "Is the comment attacking or defending the group(s) it refers to? "
                   "0 = strongly defending, 4 = strongly attacking.",
                   "a single digit 0, 1, 2, 3 or 4"),
   verb={i: f"attack-defend rating {i} of 4" for i in range(5)}),
}

RUN = ["HSBrexit", "HSBrexitOff", "ArMIS", "ConvAbuse", "MDAgreement", "AmbiStory"]
# acrescente os "MHS*" quando quiser pagar o download e as chamadas

## 3. Chamadas aos modelos

In [ ]:
TRANSIENT = {401, 408, 429, 500, 502, 503, 504}   # 401 do Cloudflare costuma ser carga, nao auth

def _post(url, payload, retries=6):
    backoff = 1.0
    last = None
    for _ in range(retries):
        try:
            r = requests.post(url, headers=HEADERS, json=payload, timeout=180)
        except requests.RequestException as e:
            last = f"rede: {e}"; time.sleep(backoff); backoff *= 2; continue
        if r.ok:
            return r.json()
        last = f"HTTP {r.status_code}: {r.text[:200]}"
        if r.status_code in TRANSIENT:
            time.sleep(float(r.headers.get("Retry-After", backoff))); backoff *= 2; continue
        raise RuntimeError(last)                   # 400, 404: erro de verdade, falha rapido
    raise RuntimeError(f"desisti apos {retries} tentativas. Ultima: {last}")

def _content(data):
    """Só o content. NUNCA cai para reasoning_content: raciocínio truncado viraria
       hedge fantasma com cobertura 100% e nada acusaria."""
    res = data.get("result", data)
    if isinstance(res, dict) and res.get("response") is not None:
        return str(res["response"])
    ch = (res.get("choices") or [{}])[0] or {}
    content = (ch.get("message", {}) or {}).get("content")
    if not content:
        raise RuntimeError(f"content vazio (finish_reason={ch.get('finish_reason')}); "
                           "provável truncamento — suba max_completion_tokens")
    return str(content)

def annotate_once(text, temperature, bench):
    b = BENCHMARKS[bench]
    out = _post(ANNOTATOR_URL, {
        "messages": [{"role": "system", "content": b["ann"]},
                     {"role": "user",   "content": f"ITEM:\n{text}"}],
        "temperature": temperature, "max_completion_tokens": 4,
        "chat_template_kwargs": {"enable_thinking": False}})
    m = re.search(r"\d", _content(out))
    if not m: return None
    v = int(m.group())
    return v if v in b["levels"] else None

def draws_variance(draws):
    """Variância das R extrações. Binário -> p(1-p); ordinal -> a mesma expressão."""
    v = [d for d in draws if d is not None]
    return float(np.var(np.asarray(v, dtype=float))) if v else float("nan")

def run_annotator(frame, bench, chunk=100):
    """Grava a cada `chunk` itens e retoma do cache. Uma queda no meio nao custa a passada."""
    cache = f"{WORKDIR}/llm1_{bench}.csv"
    done = set(pd.read_csv(cache).item_id) if os.path.exists(cache) else set()
    todo = frame[~frame.item_id.isin(done)]
    if done:
        print(f"  {bench}: {len(done)} ja em cache, faltam {len(todo)}")

    def one(row):
        try:
            lab = annotate_once(row.text, 0.0, bench)
            with ThreadPoolExecutor(max_workers=4) as ex:
                draws = list(ex.map(lambda _: annotate_once(row.text, TEMP_RESAMPLE, bench), range(R)))
        except Exception as e:                      # item problematico nao derruba o bloco
            print(f"  [{row.item_id}] {str(e)[:90]}")
            return {"item_id": row.item_id, "llm1_label": None, "g_m": float("nan"),
                    "n_valid_draws": 0}
        return {"item_id": row.item_id, "llm1_label": lab, "g_m": draws_variance(draws),
                "n_valid_draws": sum(d is not None for d in draws)}

    for i in range(0, len(todo), chunk):
        part = todo.iloc[i:i + chunk]
        with ThreadPoolExecutor(max_workers=WORKERS) as ex:
            rows = list(ex.map(one, list(part.itertuples())))
        pd.DataFrame(rows).to_csv(cache, mode="a", index=False,
                                  header=not os.path.exists(cache))
        print(f"  {bench}: {min(i + chunk, len(todo))}/{len(todo)}")
    return pd.read_csv(cache)

_HEDGE_RE = [(h, re.compile(rf"(?<!\w){re.escape(h)}(?!\w)")) for h in sorted(HEDGES, key=len, reverse=True)]

def parse_hedge(text):
    lines = [l.strip().lower().strip(".*_`\"' ") for l in (text or "").splitlines() if l.strip()]
    if lines:
        last = lines[-1]
        if last in HEDGE_TO_ORD: return HEDGE_TO_ORD[last]
        for h, rx in _HEDGE_RE:
            if rx.search(last): return HEDGE_TO_ORD[h]
    low = (text or "").lower(); best_pos, best = -1, None
    for h, rx in _HEDGE_RE:
        hits = list(rx.finditer(low))
        if hits and hits[-1].start() > best_pos: best_pos, best = hits[-1].start(), HEDGE_TO_ORD[h]
    return best

def user_msg(text, label, condition, bench):
    if condition == "labelled":
        return f"ITEM:\n{text}\n\nASSIGNED LABEL: {BENCHMARKS[bench]['verb'].get(label, label)}"
    return f"ITEM:\n{text}"

def run_assessor(codebook, fold, condition, bench):
    prompt = codebook.strip() + "\n\n" + FIXED_OUTPUT_LINE
    def one(t):
        try:
            out = _post(ASSESSOR_URL, {
                "messages": [{"role": "system", "content": prompt},
                             {"role": "user", "content": user_msg(t[0], t[1], condition, bench)}],
                "temperature": 0.0, "max_completion_tokens": 1024,
                "chat_template_kwargs": {"enable_thinking": False}})
            raw = _content(out)
            return parse_hedge(raw), raw
        except Exception as e:
            return None, f"__error__: {e}"
    with ThreadPoolExecutor(max_workers=WORKERS) as ex:
        res = list(ex.map(one, list(zip(fold["texts"], fold["labels"]))))
    return [h for h, _ in res], [o for _, o in res]

def master_lm(prompt):
    if isinstance(prompt, list):
        prompt = "\n\n".join(m.get("content", "") if isinstance(m, dict) else str(m) for m in prompt)
    return _content(_post(MASTER_URL, {"messages": [{"role": "user", "content": prompt}],
                                       "max_completion_tokens": 8000, "temperature": 1.0,
                                       "reasoning_effort": MASTER_EFFORT})).strip()

## 4. Métricas, folds e teto da escala

In [ ]:
def spearman(h, t):
    p = [(a, b) for a, b in zip(h, t) if a is not None]
    if len(p) < 2: return 0.0
    r = spearmanr(*zip(*p)).correlation
    return 0.0 if (r is None or math.isnan(r)) else float(r)

def spearman_ci(h, t, n_boot=None, seed=SEED):
    n_boot = n_boot or N_BOOT
    p = [(a, b) for a, b in zip(h, t) if a is not None]
    if len(p) < 3: return (float("nan"), float("nan"))
    hs, ts = map(np.asarray, zip(*p)); rng = np.random.default_rng(seed); B = []
    for _ in range(n_boot):
        ix = rng.integers(0, len(hs), len(hs))
        if len(set(hs[ix])) < 2 or len(set(ts[ix])) < 2: continue
        B.append(spearmanr(hs[ix], ts[ix]).correlation)
    B = [b for b in B if not math.isnan(b)]
    return tuple(np.percentile(B, [2.5, 97.5])) if B else (float("nan"), float("nan"))

def coverage(h): return sum(x is not None for x in h) / max(1, len(h))

def rho_max_discrete(values, H=H):
    """Teto exato da eq. (1) do Apêndice A.3, por programação dinâmica O(K^2 H)."""
    v = [x for x in values if not (isinstance(x, float) and math.isnan(x))]
    cnt = collections.Counter(np.round(v, 9)); vals = sorted(cnt); n = len(v)
    pi = [cnt[x] / n for x in vals]; K = len(pi)
    den = 1 - sum(p ** 3 for p in pi)
    if den <= 0: return K, float("nan")
    P = np.concatenate([[0.0], np.cumsum(pi)]); INF = float("inf")
    dp = [[INF] * (K + 1) for _ in range(H + 1)]; dp[0][0] = 0.0
    for j in range(1, H + 1):
        for i in range(1, K + 1):
            dp[j][i] = min((dp[j-1][t] + (P[i]-P[t])**3 for t in range(j-1, i) if dp[j-1][t] < INF),
                           default=INF)
    return K, math.sqrt((1 - min(dp[j][K] for j in range(1, H + 1))) / den)

def train_fold(sub):
    """Fold para o GEPA: nada de origem humana, so o que o LLM 1 produziu."""
    return {"texts": sub.text.tolist(), "labels": sub.llm1_label.tolist(),
            "gm": sub.g_m.tolist(), "ids": sub.item_id.tolist()}

def test_fold_with_human(sub):
    """Fold da avaliacao final: acrescenta o desacordo humano e o rotulo de consenso."""
    return train_fold(sub) | {"um": sub.u_m.tolist(), "consensus": sub.consensus.tolist()}

def stratified_folds(frame, n, seed):
    rng = np.random.default_rng(seed); folds = [[] for _ in range(n)]
    for _, grp in frame.groupby(frame.u_m > 0):
        for j, i in enumerate(rng.permutation(grp.index.to_numpy())): folds[j % n].append(i)
    return [frame.loc[f] for f in folds if f]

## 5. Baselines e o mestre

In [ ]:
def seed_codebook(bench, cond):
    c = BENCHMARKS[bench]["construct"]
    return c + (" Judge how uncertain that label is for this item." if cond == "labelled"
                else " Judge how uncertain a coder deciding this item's label would be.")

def baselines(bench, cond):
    c = BENCHMARKS[bench]["construct"]
    B = {"single_hedge": seed_codebook(bench, cond)}
    if cond == "labelled":
        B["reclassify"] = (c + " First decide the label yourself and justify it in one line. Then "
                           "compare your conclusion with the assigned label; the harder the "
                           "disagreement is to settle, the more uncertain the label.")
        B["topk_wrong"] = (c + " List up to three reasons why the assigned label could be WRONG for "
                           "this item, then rate the uncertainty of the label in light of them.")
        B["pros_cons"]  = (c + " List up to three reasons FOR the label being right and up to three "
                           "AGAINST, then weigh them.")
    else:
        B["topk_wrong"] = (c + " List up to three readings of this item a competent coder could "
                           "defend, then rate how uncertain the decision would be.")
        B["pros_cons"]  = (c + " List up to three reasons FOR and three AGAINST the positive label, "
                           "then weigh them.")
    return B

REFLECTION_TEMPLATE = (
    "An assistant was given the instructions below to judge how uncertain an annotation is:\n"
    "---\n<curr_param>\n---\n\n"
    "Below are items, what the assistant answered, and feedback on where it went wrong:\n"
    "---\n<side_info>\n---\n\n"
    "Rewrite the instructions so the assistant judges uncertainty better. Use the examples and the "
    "feedback to work out for yourself what makes an annotation more or less uncertain in this kind "
    "of text. Write it as an annotation codebook: name the dimensions of the construct, give "
    "observable indicators for each, and state how they combine. Form and structure are otherwise "
    "free.\n\nDo not write the output format: it is fixed and appended after your text. "
    "Give only the new instructions, between the --- lines.")

class AssessorAdapter:
    propose_new_texts = None
    def __init__(self, condition, bench): self.condition, self.bench = condition, bench

    def evaluate(self, batch, candidate, capture_traces=False):
        outs, scores, trajs = [], [], []
        for fold in batch:
            hedges, _ = run_assessor(candidate["codebook"], fold, self.condition, self.bench)
            cov = coverage(hedges)
            rho = spearman(hedges, fold["gm"])          # ALVO DE TREINO = g_m
            outs.append({"hedges": hedges})
            scores.append(rho if cov >= COVERAGE_FLOOR else 0.0)
            if capture_traces:
                gmax = max([g for g in fold["gm"] if not math.isnan(g)] or [1e-9]) or 1e-9
                worst = []
                for t, lab, h, g in zip(fold["texts"], fold["labels"], hedges, fold["gm"]):
                    d = 9.9 if h is None else abs(h / (H - 1) - g / gmax)
                    worst.append((d, t, lab, h, g))
                worst.sort(key=lambda r: -r[0])
                trajs.append({"rho": rho, "cov": cov, "worst": worst[:8]})
        return EvaluationBatch(outputs=outs, scores=scores,
                               trajectories=trajs if capture_traces else None)

    def make_reflective_dataset(self, candidate, eval_batch, components_to_update):
        data = {}
        for comp in components_to_update:
            rows = []
            for tr in eval_batch.trajectories:
                head = (f"Rank correlation on this fold: {tr['rho']:.3f} (coverage {tr['cov']:.0%}).\n"
                        "The items below are where the assessor diverged most from how unstable the "
                        "annotator actually was across repeated draws.\n")
                for d, t, lab, h, g in tr["worst"]:
                    said = HEDGES[h] if isinstance(h, int) else "NO VALID HEDGE"
                    truth = ("the annotator varied across draws" if g > 0
                             else "the annotator returned the same label on every draw")
                    rows.append({"Inputs": f"ITEM: {t[:400]}\nASSIGNED LABEL: {lab}",
                                 "Generated Outputs": said,
                                 "Feedback": head + f"On this item {truth}, but the assessor said '{said}'."})
            data[comp] = rows
        return data

## 6. Custo antes de gastar

Roda primeiro. Nada de rede além do carregamento dos corpora.

In [ ]:
DATA = {}
for b in RUN:
    DATA[b] = BENCHMARKS[b]["loader"]().reset_index(drop=True)   # corpus completo

est = []
for b, d in DATA.items():
    n = len(d); ntr = (d.split_orig == "train").sum(); nte = (d.split_orig == "test").sum()
    n_meth = len(baselines(b, "labelled")) + len(baselines(b, "blind")) + 2   # +GEPA por condição
    n_annot = n * (R + 1)
    n_gepa  = MAX_METRIC_CALLS * (ntr / N_TRAIN_FOLDS) * len(CONDITIONS)
    n_eval  = nte * n_meth
    est.append({"benchmark": b, "itens": n, "u_m>0": f"{(d.u_m>0).mean():.0%}",
                "chamadas LLM1": int(n_annot), "chamadas GEPA": int(n_gepa),
                "chamadas aval.": int(n_eval), "total": int(n_annot + n_gepa + n_eval)})
cost = pd.DataFrame(est)
print(cost.to_string(index=False))
print(f"\nTOTAL GERAL: {cost.total.sum():,} chamadas "
      f"({cost['chamadas LLM1'].sum():,} do LLM 1, pagas uma vez pelo cache em disco)")

  benchmark  itens u_m>0  chamadas LLM1  chamadas GEPA  chamadas aval.  total
   HSBrexit   1120   31%          23520          15679            1512  40712
HSBrexitOff   1117   47%          23457          15640            1503  40600
      ArMIS    943   35%          19803          13140            1305  34248
  ConvAbuse   4050   20%          85050          47960            7560 140570
MDAgreement  10753   58%         225813         131840           27513 385166
  AmbiStory   3798   91%          79758          45600            8370 133728

TOTAL GERAL: 775,024 chamadas (457,401 do LLM 1, pagas uma vez pelo cache em disco)


## 7. LLM 1 em todos os benchmarks (com cache em disco)

In [ ]:
FRAMES = {}
for b in RUN:
    print(f"{b}: LLM 1 em {len(DATA[b])} itens")
    llm1 = run_annotator(DATA[b], b)
    f = DATA[b].merge(llm1, on="item_id").dropna(subset=["llm1_label", "g_m"]).reset_index(drop=True)
    f["llm1_label"] = f.llm1_label.astype(int)
    FRAMES[b] = f
    acc = (f.llm1_label == f.consensus).mean()
    base = f.consensus.value_counts(normalize=True).max()
    print(f"  n={len(f)} acc={acc:.3f} (baseline {base:.3f}) "
          f"g_m=0 em {(f.g_m==0).mean():.1%} | rho(g,u)={spearman(f.g_m, f.u_m):+.3f}")


HSBrexit: LLM 1 em 1120 itens
  HSBrexit: 1120 ja em cache, faltam 0
  n=1120 acc=0.856 (baseline 0.942) g_m=0 em 97.1% | rho(g,u)=+0.153
HSBrexitOff: LLM 1 em 1117 itens
  HSBrexitOff: 1117 ja em cache, faltam 0
  n=1117 acc=0.842 (baseline 0.860) g_m=0 em 93.4% | rho(g,u)=+0.194
ArMIS: LLM 1 em 943 itens
  ArMIS: 943 ja em cache, faltam 0
  n=943 acc=0.696 (baseline 0.587) g_m=0 em 86.0% | rho(g,u)=+0.114
ConvAbuse: LLM 1 em 4050 itens
  ConvAbuse: 4050 ja em cache, faltam 0
  n=4050 acc=0.908 (baseline 0.783) g_m=0 em 96.4% | rho(g,u)=+0.153
MDAgreement: LLM 1 em 10753 itens
  MDAgreement: 10753 ja em cache, faltam 0
  n=10753 acc=0.810 (baseline 0.687) g_m=0 em 91.3% | rho(g,u)=+0.164
AmbiStory: LLM 1 em 3798 itens
  AmbiStory: 3798 ja em cache, faltam 0
  n=3798 acc=0.304 (baseline 0.245) g_m=0 em 72.2% | rho(g,u)=+0.168


## 8. GEPA por benchmark e condição

In [ ]:
print(master_lm("Reply with one short sentence: this is a smoke test."))

This smoke test is working correctly.


In [ ]:
assert hasattr(gepa, "optimize"), f"'gepa' virou {type(gepa)}; reimporte o modulo"

CODEBOOKS = {}
for b in RUN:
    f = FRAMES[b]
    tr = [train_fold(x) for x in stratified_folds(f[f.split_orig=="train"], N_TRAIN_FOLDS, SEED)]
    va = [train_fold(x) for x in stratified_folds(f[f.split_orig=="dev"],   N_VAL_FOLDS,  SEED+1)]
    for cond in CONDITIONS:
        key = f"{b}__{cond}"; path = f"{WORKDIR}/codebook_{key}.txt"
        if os.path.exists(path):
            CODEBOOKS[key] = open(path, encoding="utf-8").read(); print(f"{key}: cache"); continue
        print(f"\n=== {key} ===")
        res = gepa.optimize(seed_candidate={"codebook": seed_codebook(b, cond)},
                            trainset=tr, valset=va, adapter=AssessorAdapter(cond, b),
                            reflection_lm=master_lm, candidate_selection_strategy="pareto",
                            reflection_prompt_template=REFLECTION_TEMPLATE,
                            reflection_minibatch_size=1, max_metric_calls=MAX_METRIC_CALLS,
                            display_progress_bar=True, raise_on_exception=True, seed=SEED)
        CODEBOOKS[key] = res.best_candidate["codebook"]
        open(path, "w", encoding="utf-8").write(CODEBOOKS[key])


=== HSBrexit__labelled ===


GEPA Optimization:   3%|▎         | 2/60 [00:13<06:44,  6.97s/rollouts]

Iteration 0: Base program full valset score: 0.0719044948576776 over 2 / 2 examples
Iteration 1: Selected program 0 score: 0.0719044948576776
Iteration 1: Proposed new text for codebook: ---
# Annotation Codebook: Judging Annotator Uncertainty for Hate Speech Labels

Your task is to judge how uncertain a human annotator would be when labeling an English tweet about Brexit and immigration for HATE SPEECH (language that attacks, demeans, or excludes people because of their religion, national origin, or migration status). 

Annotator uncertainty reflects how likely it is that the annotator would provide different labels if asked to annotate the same tweet multiple times. To judge this, you must evaluate the text for features that typically cause ambiguity or confusion in hate speech annotation.

## Dimensions of Uncertainty

### 1. Explicitness of Attack
**Indicator:** How direct and unambiguous is the language regarding attacks, demeaning, or exclusion?
- **High Uncertainty:** The tweet 

GEPA Optimization:  10%|█         | 6/60 [01:06<10:20, 11.50s/rollouts]

Iteration 1: Accepted candidate (subsample score -0.1294507743629427 -> 0.09155783429543661); running full eval.
Iteration 1: Found a better program on the valset with score 0.17805467717584433.
Iteration 1: Valset score for new program: 0.17805467717584433 (coverage 2 / 2)
Iteration 1: Val aggregate for new program: 0.17805467717584433
Iteration 1: Individual valset scores for new program: {0: 0.13047391948807915, 1: 0.2256354348636095}
Iteration 1: New valset pareto front scores: {0: 0.13047391948807915, 1: 0.2256354348636095}
Iteration 1: Valset pareto front aggregate score: 0.17805467717584433
Iteration 1: Updated valset pareto front programs: {0: {1}, 1: {1}}
Iteration 1: Best valset aggregate score so far: 0.17805467717584433
Iteration 1: Best program as per aggregate score on valset: 1
Iteration 1: Best score on valset: 0.17805467717584433
Iteration 1: Linear pareto front program index: 1
Iteration 1: New program candidate index: 1
Iteration 2: Selected program 1 score: 0.178054

...

Iteration 20: Accepted candidate (subsample score -0.03336282661540022 -> 0.0); running full eval.
Iteration 20: Valset score for new program: 0.0 (coverage 2 / 2)
Iteration 20: Val aggregate for new program: 0.0
Iteration 20: Individual valset scores for new program: {0: 0.0, 1: 0.0}
Iteration 20: New valset pareto front scores: {0: 0.21528080956231058, 1: 0.22387269875288834}
Iteration 20: Valset pareto front aggregate score: 0.21957675415759947
Iteration 20: Updated valset pareto front programs: {0: {3}, 1: {4}}
Iteration 20: Best valset aggregate score so far: 0.18786036491907693
Iteration 20: Best program as per aggregate score on valset: 3
Iteration 20: Best score on valset: 0.18786036491907693
Iteration 20: Linear pareto front program index: 3
Iteration 20: New program candidate index: 7
Iteration 21: Selected program 4 score: 0.1752866518951487
Iteration 21: Proposed new text for codebook: # Uncertainty Assessment Codebook

Your task is to judge how uncertain an annotator would

GEPA Optimization:  97%|█████████▋| 58/60 [1:36:57<03:41, 110.53s/rollouts]

Iteration 21: New subsample score 0.07170177484482358 is not better than old score 0.1148138095476325, skipping
Iteration 22: Selected program 3 score: 0.18786036491907693
Iteration 22: Proposed new text for codebook: ---
# Uncertainty Assessment Codebook

Your task is to judge how uncertain an annotator would be when rating a candidate meaning for an ambiguous word in a short narrative.

## CRITICAL OUTPUT RULES
- **You must ALWAYS output exactly one of these exact hedges: "highly certain", "moderately certain", or "highly uncertain".**
- **NEVER output "NO VALID HEDGE", "not uncertain", "uncertain", "refuse", or any other phrase.** You must always evaluate the context and pick the closest fit. There is no situation where you should refuse to provide a hedge.
- **Extreme labels (1 or 5) do NOT imply uncertainty.** A label of 1 can be just as certain as a label of 5. If the context definitively rules out the candidate meaning, the annotator is highly certain even with a label of 1.

##

GEPA Optimization:  97%|█████████▋| 58/60 [1:40:11<03:27, 103.65s/rollouts]

Iteration 22: New subsample score 0.0 is not better than old score 0.1741596345930657, skipping


## 10. Avaliação no teste, N execuções por método

In [12]:
rows, per_item = [], []
for b in RUN:
    f = FRAMES[b]; test = test_fold_with_human(f[f.split_orig == "test"])
    K_u, ceil_u = rho_max_discrete(test["um"]); K_g, ceil_g = rho_max_discrete(test["gm"])
    for cond in CONDITIONS:
        meths = dict(baselines(b, cond)); meths["GEPA"] = CODEBOOKS[f"{b}__{cond}"]
        for name, prompt in meths.items():
            h, raw = run_assessor(prompt, test, cond, b)
            per_item.append(pd.DataFrame({
                "benchmark": b, "condition": cond, "method": name,
                "item_id": test["ids"], "text": test["texts"],
                "llm1_label": test["labels"], "consensus": test["consensus"],
                "g_m": test["gm"], "u_m": test["um"],
                "hedge": [HEDGES[x] if isinstance(x, int) else None for x in h], "raw": raw}))
            ru = spearman(h, test["um"]); rg = spearman(h, test["gm"])
            lo, hi = spearman_ci(h, test["um"])
            dist = collections.Counter(x for x in h if x is not None)
            rows.append({
                "benchmark": b, "condition": cond, "method": name,
                "rho_u": ru, "ci_lo": lo, "ci_hi": hi, "ceil_u": ceil_u, "pct_ceil": ru / ceil_u,
                "rho_g": rg, "ceil_g": ceil_g, "coverage": coverage(h),
                "levels_used": len(dist), "modal_share": (max(dist.values()) / sum(dist.values())
                                                          if dist else float("nan")),
                "n_test": len(test["ids"]), "K_u": K_u})
            print(f"{b:14} {cond:9} {name:14} rho_u={ru:+.3f} [{lo:+.2f},{hi:+.2f}]  "
                  f"rho_g={rg:+.3f}  níveis={len(dist)}  cov={coverage(h):.0%}")

summary = pd.DataFrame(rows); summary.to_csv(f"{RESDIR}/summary.csv", index=False)
pd.concat(per_item).to_csv(f"{RESDIR}/per_item.csv", index=False)
summary

HSBrexit       labelled  single_hedge   rho_u=+0.224 [+0.14,+0.29]  rho_g=+0.098  níveis=2  cov=100%
HSBrexit       labelled  reclassify     rho_u=-0.070 [-0.12,-0.04]  rho_g=-0.023  níveis=2  cov=85%
HSBrexit       labelled  topk_wrong     rho_u=+0.116 [-0.05,+0.28]  rho_g=+0.047  níveis=2  cov=100%
HSBrexit       labelled  pros_cons      rho_u=+0.228 [+0.06,+0.39]  rho_g=+0.140  níveis=3  cov=99%
HSBrexit       labelled  GEPA           rho_u=+0.342 [+0.17,+0.50]  rho_g=+0.157  níveis=2  cov=100%
HSBrexit       blind     single_hedge   rho_u=+0.078 [-0.07,+0.23]  rho_g=+0.071  níveis=3  cov=100%
HSBrexit       blind     topk_wrong     rho_u=+0.197 [+0.04,+0.35]  rho_g=-0.018  níveis=3  cov=100%
HSBrexit       blind     pros_cons      rho_u=+0.200 [+0.03,+0.36]  rho_g=-0.082  níveis=3  cov=100%
HSBrexit       blind     GEPA           rho_u=+0.467 [+0.33,+0.59]  rho_g=+0.176  níveis=3  cov=100%
HSBrexitOff    labelled  single_hedge   rho_u=+0.285 [+0.18,+0.38]  rho_g=+0.091  níveis=2  c

,benchmark,condition,method,rho_u,ci_lo,ci_hi,ceil_u,pct_ceil,rho_g,ceil_g,coverage,levels_used,modal_share,n_test,K_u
0,HSBrexit,labelled,single_hedge,0.223945,0.139334,0.294627,0.995924,0.224862,0.097599,0.999928,1.000000,2,0.869048,168,4
1,HSBrexit,labelled,reclassify,-0.070001,-0.119976,-0.043233,0.995924,-0.070288,-0.022829,0.999928,0.845238,2,0.985915,168,4
2,HSBrexit,labelled,topk_wrong,0.115729,-0.048225,0.284024,0.995924,0.116202,0.047190,0.999928,1.000000,2,0.946429,168,4
3,HSBrexit,labelled,pros_cons,0.227702,0.060187,0.386624,0.995924,0.228634,0.139814,0.999928,0.988095,3,0.801205,168,4
4,HSBrexit,labelled,GEPA,0.342183,0.172072,0.501726,0.995924,0.343583,0.156820,0.999928,1.000000,2,0.827381,168,4
5,HSBrexit,blind,single_hedge,0.077964,-0.070898,0.228296,0.995924,0.078283,0.070637,0.999928,1.000000,3,0.726190,168,4
6,HSBrexit,blind,topk_wrong,0.196618,0.036767,0.347051,0.995924,0.197423,-0.018253,0.999928,1.000000,3,0.505952,168,4
7,HSBrexit,blind,pros_cons,0.199952,0.027872,0.359266,0.995924,0.200770,-0.081529,0.999928,1.000000,3,0.904762,168,4
8,HSBrexit,blind,GEPA,0.466705,0.328900,0.591630,0.995924,0.468616,0.176037,0.999928,1.000000,3,0.583333,168,4
9,HSBrexitOff,labelled,single_hedge,0.284612,0.184148,0.375915,0.986860,0.288401,0.091141,0.999686,1.000000,2,0.922156,167,4


## 11. Tabelas e placeholders do paper

Escreve `paper/*.tex`. Os números do texto entram por macro (`macros.tex`), então reexecutar
atualiza o paper inteiro sem edição manual.

In [13]:
def esc(s): return (str(s).replace("_", r"\_").replace("&", r"\&").replace("%", r"\%"))
def fmt(x, d=3): return "--" if (x is None or (isinstance(x,float) and math.isnan(x))) else f"{x:.{d}f}"

# ---- Tabela 1: descritivos dos dados ----
L = [r"\begin{table}[t]", r"\centering", r"\small",
     r"\begin{tabular}{lrrrrrr}", r"\hline",
     r"Benchmark & Items & Ann./item & \%\,$u_m{=}0$ & $K_u$ & \%\,$g_m{=}0$ & $\rho(g_m,u_m)$ \\",
     r"\hline"]
for b in RUN:
    f = FRAMES[b]; K, _ = rho_max_discrete(f.u_m)
    na = f.n_ann.min() if f.n_ann.nunique() == 1 else f"{f.n_ann.min()}--{f.n_ann.max()}"
    L.append(f"{esc(b)} & {len(f)} & {na} & {(f.u_m==0).mean()*100:.1f} & {K} & "
             f"{(f.g_m==0).mean()*100:.1f} & {fmt(spearman(f.g_m, f.u_m))} \\\\")
L += [r"\hline", r"\end{tabular}",
      r"\caption{Benchmarks. $u_m$ is human disagreement, $g_m$ the variance of the $R=20$ "
      r"annotator draws, $K_u$ the number of distinct values $u_m$ takes.}",
      r"\label{tab:data}", r"\end{table}"]
open(f"{PAPERDIR}/tab_data.tex", "w").write("\n".join(L))

# ---- Tabela 2: resultado principal (condição blind), rho_u por benchmark x método ----
def result_table(cond, fname, cap):
    piv = (summary[summary.condition == cond]
           .pivot(index="method", columns="benchmark", values="rho_u"))
    order = [m for m in ["single_hedge","reclassify","topk_wrong","pros_cons","GEPA"] if m in piv.index]
    piv = piv.loc[order, [b for b in RUN if b in piv.columns]]
    L = [r"\begin{table}[t]", r"\centering", r"\small",
         r"\begin{tabular}{l" + "r"*len(piv.columns) + "}", r"\hline",
         "Method & " + " & ".join(esc(c) for c in piv.columns) + r" \\", r"\hline"]
    for m in piv.index:
        cellz = []
        for c in piv.columns:
            v = piv.loc[m, c]
            best = v == piv[c].max()
            cellz.append((r"\textbf{" + fmt(v) + "}") if best else fmt(v))
        L.append(esc(m) + " & " + " & ".join(cellz) + r" \\")
    ceil_row = [fmt(summary[(summary.condition==cond)&(summary.benchmark==c)].ceil_u.iloc[0])
                for c in piv.columns]
    L += [r"\hline", r"$\rho_{\max}(H{=}3)$ & " + " & ".join(ceil_row) + r" \\",
          r"\hline", r"\end{tabular}", r"\caption{" + cap + r"}",
          r"\label{tab:main-" + cond + r"}", r"\end{table}"]
    open(f"{PAPERDIR}/{fname}", "w").write("\n".join(L))

result_table("blind", "tab_main.tex",
             r"Spearman $\rho(h_m,u_m)$ on held-out test items, message-only condition. "
             r"Best per column in bold.")

# ---- Tabela 3: ablation labelled vs blind ----
L = [r"\begin{table}[t]", r"\centering", r"\small", r"\begin{tabular}{llrrr}", r"\hline",
     r"Benchmark & Method & labelled & message-only & $\Delta$ \\", r"\hline"]
for b in RUN:
    for m in ["single_hedge", "topk_wrong", "pros_cons", "GEPA"]:
        s = summary[(summary.benchmark==b) & (summary.method==m)]
        if set(s.condition) != {"labelled","blind"}: continue
        a = s[s.condition=="labelled"].rho_u.iloc[0]; c = s[s.condition=="blind"].rho_u.iloc[0]
        L.append(f"{esc(b)} & {esc(m)} & {fmt(a)} & {fmt(c)} & {fmt(a-c,3)} \\\\")
L += [r"\hline", r"\end{tabular}",
      r"\caption{Assessor input ablation. $\Delta>0$ means the assigned label helped.}",
      r"\label{tab:ablation}", r"\end{table}"]
open(f"{PAPERDIR}/tab_ablation.tex", "w").write("\n".join(L))

# ---- Tabela 4: estabilidade entre execuções ----
L = [r"\begin{table}[t]", r"\centering", r"\small", r"\begin{tabular}{llrrr}", r"\hline",
     r"Benchmark & Method & Levels & Modal share & Coverage \\", r"\hline"]
for _, r_ in summary[summary.condition=="blind"].iterrows():
    L.append(f"{esc(r_.benchmark)} & {esc(r_.method)} & {int(r_.levels_used)} & "
             f"{fmt(r_.modal_share,2)} & {fmt(r_.coverage,2)} \\\\")
L += [r"\hline", r"\end{tabular}",
      r"\caption{Use of the hedge scale. Levels is how many of the $H=3$ hedges a method ever "
      r"emits; modal share is the fraction of items receiving the most common one. A method that "
      r"collapses the scale is capped well below $\rho_{\max}$ regardless of how well it ranks.}",
      r"\label{tab:stability}", r"\end{table}"]
open(f"{PAPERDIR}/tab_stability.tex", "w").write("\n".join(L))

# ---- Apêndice: codebooks ----
L = [r"\section{Optimised codebooks}", r"\label{app:codebooks}"]
for k, cb in sorted(CODEBOOKS.items()):
    b, cond = k.split("__")
    L += [r"\subsection*{" + esc(b) + " -- " + esc(cond) + "}", r"\begin{quote}\small",
          esc(cb).replace("\n", "\n\n"), r"\end{quote}"]
open(f"{PAPERDIR}/codebooks.tex", "w").write("\n".join(L))

# ---- Macros ----
def mac(name): return "".join(w.capitalize() for w in re.split(r"[^A-Za-z]+", name) if w)
M = [r"% gerado por 10_all_benchmarks.ipynb -- nao editar a mao"]
for _, r_ in summary.iterrows():
    base = mac(r_.benchmark) + mac(r_.condition) + mac(r_.method)
    M.append(r"\providecommand{\rhoU" + base + "}{" + fmt(r_.rho_u) + "}")
    M.append(r"\providecommand{\ciU" + base + "}{[" + fmt(r_.ci_lo,2) + ", " + fmt(r_.ci_hi,2) + "]}")
for b in RUN:
    f = FRAMES[b]
    M += [r"\providecommand{\n" + mac(b) + "}{" + str(len(f)) + "}",
          r"\providecommand{\accLLMone" + mac(b) + "}{" + fmt((f.llm1_label==f.consensus).mean()) + "}",
          r"\providecommand{\gzero" + mac(b) + "}{" + f"{(f.g_m==0).mean()*100:.1f}" + "}",
          r"\providecommand{\rhoGU" + mac(b) + "}{" + fmt(spearman(f.g_m, f.u_m)) + "}"]
M += [r"\providecommand{\Rdraws}{" + str(R) + "}",
      r"\providecommand{\nHedges}{" + str(H) + "}"]
open(f"{PAPERDIR}/macros.tex", "w").write("\n".join(M))

# ---- Esqueleto da seção de resultados ----
best = summary[(summary.condition=="blind") & (summary.method=="GEPA")]
sk = [r"% \input{paper/macros} no preambulo",
      r"\section{Results}", "",
      r"Table~\ref{tab:data} describes the benchmarks. \textbf{[TODO: uma frase sobre a faixa de "
      r"$\rho(g_m,u_m)$ e o que ela implica.]}", "",
      r"\input{paper/tab_data}", "",
      r"Table~\ref{tab:main-blind} reports the headline association. On "
      + esc(RUN[0]) + r" the optimised codebook reaches $\rho = \rhoU"
      + mac(RUN[0]) + r"BlindGepa$ (\ciU" + mac(RUN[0]) + r"BlindGepa), against "
      r"$\rhoU" + mac(RUN[0]) + r"BlindSingleHedge$ for a single unoptimised hedge. "
      r"\textbf{[TODO: o padrao se mantem nos demais? onde quebra?]}", "",
      r"\input{paper/tab_main}", "",
      r"\paragraph{Assessor input.} \textbf{[TODO: ler a coluna $\Delta$ da Tabela~"
      r"\ref{tab:ablation}. Se $\Delta<0$ de forma consistente, o rotulo atrapalha e a Secao~4 "
      r"precisa ser reescrita nesse ponto.]}", "",
      r"\input{paper/tab_ablation}", "",
      r"\paragraph{Use of the scale.} Table~\ref{tab:stability} reports how much of the "
      r"three-level scale each method actually uses. \textbf{[TODO: comentar se os baselines "
      r"colapsam a escala, e se a vantagem do codebook vem de ordenar melhor ou de usar a "
      r"escala inteira.]}", "",
      r"\input{paper/tab_stability}"]
open(f"{PAPERDIR}/results_skeleton.tex", "w").write("\n".join(sk))

print("escritos:", sorted(os.listdir(PAPERDIR)))

escritos: ['codebooks.tex', 'macros.tex', 'results_skeleton.tex', 'tab_ablation.tex', 'tab_data.tex', 'tab_main.tex', 'tab_stability.tex']


In [15]:
!cd . && zip -qr paper_materials.zip paper results work/codebook_*.txt && ls -la paper_materials.zip
from google.colab import files; files.download("paper_materials.zip")

-rw-r--r-- 1 root root 10058558 Sep 16 14:10 paper_materials.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>